In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install haystack-ai elasticsearch-haystack
!pip install evaluate

In [ ]:
## データセット用意
from datasets import load_dataset
subjqa = load_dataset("subjqa", name="electronics")
import pandas as pd
dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

In [ ]:
## Elasticsearch ダウンロード
url = "https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.17.10-linux-x86_64.tar.gz"
!wget -nc -q {url}
!tar -xzf elasticsearch-7.17.10-linux-x86_64.tar.gz

In [ ]:
## Elasticsearch 起動
!pkill -f org.elasticsearch.bootstrap.Elasticsearch || true
!sysctl -w vm.max_map_count=262144
!chown -R daemon:daemon /content/elasticsearch-7.17.10
!su -s /bin/bash -c 'export ES_JAVA_OPTS="-Des.cgroups.hierarchy.override=/ -XX:-UseContainerSupport"; \
/content/elasticsearch-7.17.10/bin/elasticsearch \
  -Ediscovery.type=single-node \
  -Expack.security.enabled=false \
  -Expack.ml.enabled=false \
  -Ehttp.host=127.0.0.1 \
  -Etransport.host=127.0.0.1 \
  > /tmp/es.log 2>&1 &' daemon
!sleep 25
!tail -n 120 /tmp/es.log
!curl -s http://127.0.0.1:9200

In [ ]:
### 起動確認
!curl -X GET "localhost:9200/?pretty"

In [ ]:
## Retriever の用意
### ドキュメントストアのインスタンス化
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
custom_mapping = {
    "properties": {
        "content": {"type": "text"},
        "id": {"type": "keyword"},
        "item_id": {"type": "keyword"},
        "question_id": {"type": "keyword"},
        "split": {"type": "keyword"}
    }
}
document_store = ElasticsearchDocumentStore(hosts="http://localhost:9200",
                                            index="my_document_store",
                                            custom_mapping=custom_mapping)

### Haystack でのドキュメントストアのインデックス化
from haystack.dataclasses import Document
from haystack.document_stores.types import DuplicatePolicy
import json
def safe_str(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    if isinstance(v, (dict, list, tuple)):
        return json.dumps(v, ensure_ascii=False)
    return str(v)
for split, df in dfs.items():
    # 重複するレビュー除去
    work = df.dropna(subset=["context"]).drop_duplicates(subset="context")
    docs = [
        Document(
            content=safe_str(row.get("context")).strip(),
            meta={
                "item_id": safe_str(row.get("title")),
                "question_id": safe_str(row.get("id")),
                "split": safe_str(split),
            },
        )
        for _, row in work.iterrows()
        if safe_str(row.get("context")).strip()
    ]
    if docs:
        written = document_store.write_documents(
            docs,
            policy=DuplicatePolicy.SKIP,
            refresh="wait_for",
        )
print(f"Loaded {document_store.count_documents()} documents")

### Elastic Search を用いたレトリーバー
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
es_retriever = ElasticsearchBM25Retriever(document_store=document_store)

In [ ]:
# Label ドキュメントのリスト作成
from haystack import Document
from haystack.document_stores.types import DuplicatePolicy
label_store = ElasticsearchDocumentStore(hosts="http://localhost:9200",
                                            index="label",
                                            custom_mapping=custom_mapping)

label_docs = []
no_ans_count = 0
for i, row in dfs["test"].iterrows():
    qid = str(row["id"])

    if not len(row["answers.text"]):
        label_docs.append(
            Document(
                content="",
                meta={
                    "type": "label",
                    "question_id": qid,
                    "item_id": row["title"],
                    "question": row["question"],
                    "answer": "",
                    "no_answer": True,
                    "row_id": int(i),
                },
            )
        )
        no_ans_count += 1
    else:
        for ans in row["answers.text"]:
            label_docs.append(
                Document(
                    content=ans,
                    meta={
                        "type": "label",
                        "question_id": qid,
                        "item_id": row["title"],
                        "question": row["question"],
                        "answer": ans,
                        "no_answer": False,
                        "row_id": int(i),
                    },
                )
            )

print(f"loaded data: {len(label_docs)}, no_ans data: {no_ans_count}")
label_store.write_documents(label_docs, policy=DuplicatePolicy.SKIP)

In [ ]:
# ラベル確認
docs = label_store.filter_documents()
print(docs[0])

In [ ]:
# item_id で集約
from collections import defaultdict
from haystack import component

@component
class LabelAggregatorByItem:
    def __init__(self, label_document_store):
        self.store = label_document_store

    @component.output_types(relevant_item_ids=list[str], answers_by_item=dict)
    def run(self, question_id: str):
        qid = str(question_id).strip()

        docs = self.store.filter_documents(
            filters={
                "operator": "AND",
                "conditions": [
                    {"field": "meta.question_id", "operator": "==", "value": qid},
                    {"field": "meta.no_answer",   "operator": "==", "value": False},
                ],
            }
        )

        agg = defaultdict(set)
        for d in docs:
            item_id = str(d.meta.get("item_id", "")).strip()
            answer  = str(d.meta.get("answer",  "")).strip()
            if item_id and answer:
                agg[item_id].add(answer)

        answers_by_item   = {k: sorted(v) for k, v in agg.items()}
        relevant_item_ids = sorted(answers_by_item.keys())
        return {"relevant_item_ids": relevant_item_ids, "answers_by_item": answers_by_item}

In [ ]:
from elasticsearch._sync.client import Elasticsearch
# 評価値計算
@component
class ItemRecallEvaluator:
    """
    Retriever 出力 documents の item_id と
    label 側 relevant_item_ids を比較して Recall@k を返す
    """
    @component.output_types(recall=float, hits=int, total_relevant=int)
    def run(self, documents: list, relevant_item_ids: list[str]):
        retrieved_item_ids = []
        for d in documents:
            item_id = d.meta.get("item_id")
            if item_id is not None:
                retrieved_item_ids.append(str(item_id))

        retrieved_set = set(retrieved_item_ids)
        relevant_set = set(relevant_item_ids)

        if not relevant_set:
            return {"recall": 0.0, "hits": 0, "total_relevant": 0}

        hits = len(retrieved_set & relevant_set)
        recall = hits / len(relevant_set)
        return {"recall": recall, "hits": hits, "total_relevant": len(relevant_set)}

# Pipeline 構築
from haystack import Pipeline
retriever = ElasticsearchBM25Retriever(document_store=document_store)

pipe = Pipeline()
pipe.add_component("Retriever", retriever)
pipe.add_component("LabelAgg", LabelAggregatorByItem(label_store))
pipe.add_component("Eval", ItemRecallEvaluator())

pipe.connect("Retriever.documents", "Eval.documents")
pipe.connect("LabelAgg.relevant_item_ids", "Eval.relevant_item_ids")

In [ ]:
# スコア計算
def run_eval_pipeline(pipeline, top_k):
  scores = []
  empty_rel = 0
  for _, row in dfs["test"].iterrows():
      qid = str(row["id"])
      question = safe_str(row["question"])

      out = pipeline.run(
          data={
              "Retriever": {"query": question, "top_k": top_k},
              "LabelAgg": {"question_id": qid},
          }
      )

      ev = out["Eval"]
      if ev["total_relevant"] == 0:
          empty_rel += 1
          continue
      scores.append(ev["recall"])

  return scores, empty_rel

scores, empty_rel = run_eval_pipeline(pipe, 10)

print("evaluated:", len(scores))
print("empty_relevant:", empty_rel)
print("Recall@10:", (sum(scores)/len(scores)) if scores else None)

In [ ]:
# k についてチェック
def evaluate_retriever(pipeline, topk_values = [1, 3, 5, 10, 20]):
  topk_results = {}

  for topk in topk_values:
    scores, empty_relevant = run_eval_pipeline(pipeline, topk)
    topk_results[topk] = {"recall": (sum(scores)/len(scores))}
    print(f"top_k: {topk}, empty_relevant: {empty_relevant}")

  return pd.DataFrame.from_dict(topk_results, orient="index")

es_topk_results = evaluate_retriever(pipe)

In [ ]:
# 結果描画
import matplotlib.pyplot as plt
def plot_retriever_eval(dfs, retriever_names):
  fig, ax = plt.subplots()
  for df, retriever_name in zip(dfs, retriever_names):
    df.plot(y="recall", ax=ax, label=retriever_name)
  plt.xticks(df.index)
  plt.ylabel("Top-k Recall")
  plt.xlabel("k")
  plt.show()

plot_retriever_eval([es_topk_results], ["BM25"])

In [ ]:
from haystack_integrations.components.retrievers.elasticsearch.embedding_retriever import ElasticsearchEmbeddingRetriever
# DensePassageRetriever 用意
## haystack 2.x 系では EmbeddingRetriever に集約されている
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder

## 埋め込みベクトルストア
embedding_dim = 768
emb_store = ElasticsearchDocumentStore(
    hosts="http://localhost:9200",
    index="emb_store",
    custom_mapping={
        "properties": {
            "embedding": {
              "type": "dense_vector",
              "index": True,
              "dims": embedding_dim,
              "similarity": "cosine",
          },
          "content": {"type": "text",},
          "id": {"type": "keyword"},
          "item_id": {"type": "keyword"},
          "question_id": {"type": "keyword"},
          "split": {"type": "keyword"}
      }
    },
)

## 埋め込み
doc_embedder = SentenceTransformersDocumentEmbedder(
  model="sentence-transformers/multi-qa-mpnet-base-dot-v1"
)
doc_embedder.warm_up()

embedded_docs = doc_embedder.run(documents=label_docs)
emb_store.write_documents(embedded_docs["documents"], policy=DuplicatePolicy.SKIP)

## クエリ埋め込み
query_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/multi-qa-mpnet-base-dot-v1"
)

## レトリーバー
from typing import List
@component
class ES7EmbeddingRetriever:
    def __init__(self, document_store: ElasticsearchDocumentStore, top_k: int = 3):
        self.document_store = document_store
        self.top_k = top_k

    @component.output_types(documents=List[Document])
    def run(self, query_embedding: List[float], top_k: int = 0):
        k = top_k if top_k > 0 else self.top_k
        body = {
            "size": k,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.q, 'embedding') + 1.0",
                        "params": {"q": query_embedding},
                    },
                }
            },
        }
        res = self.document_store._client.search(
            index=self.document_store._index,
            body=body,
        )

        docs = []
        for hit in res["hits"]["hits"]:
            src = hit["_source"]
            meta = {kk: vv for kk, vv in src.items() if kk not in ("content", "embedding")}
            docs.append(
                Document(
                    id=src.get("id"),
                    content=src.get("content", ""),
                    meta=meta,
                    score=hit.get("_score"),
                )
            )
        return {"documents": docs}

emb_retriever = ES7EmbeddingRetriever(
    document_store=emb_store,
    top_k=3,
)

In [ ]:
# 評価パイプライン
emb_pip = Pipeline()
emb_pip.add_component("QueryEmbedder", query_embedder)
emb_pip.add_component("Retriever", emb_retriever)
emb_pip.add_component("LabelAgg", LabelAggregatorByItem(emb_store))
emb_pip.add_component("Eval", ItemRecallEvaluator())

emb_pip.connect("QueryEmbedder.embedding", "Retriever.query_embedding")
emb_pip.connect("Retriever.documents", "Eval.documents")
emb_pip.connect("LabelAgg.relevant_item_ids", "Eval.relevant_item_ids")

In [ ]:
# 結果
def run_eval_emb_pipeline(pipeline, top_k):
  scores = []
  empty_rel = 0
  for _, row in dfs["test"].iterrows():
      qid = str(row["id"])
      question = safe_str(row["question"])

      out = pipeline.run(
          data={
              "QueryEmbedder": {"text": question},
              "Retriever": {"top_k": top_k},
              "LabelAgg": {"question_id": qid},
          }
      )

      ev = out["Eval"]
      if ev["total_relevant"] == 0:
          empty_rel += 1
          continue
      scores.append(ev["recall"])

  return scores, empty_rel

# k についてチェック
def evaluate_emb_retriever(pipeline, topk_values = [1, 3, 5, 10, 20]):
  topk_results = {}

  for topk in topk_values:
    scores, empty_relevant = run_eval_emb_pipeline(pipeline, topk)
    topk_results[topk] = {"recall": (sum(scores)/len(scores))}
    print(f"top_k: {topk}, empty_relevant: {empty_relevant}")

  return pd.DataFrame.from_dict(topk_results, orient="index")

emb_topk_results = evaluate_emb_retriever(emb_pip)
plot_retriever_eval([es_topk_results, emb_topk_results], ["BM25", "DPR"])

In [ ]:
!pip install evaluate

In [ ]:
## Reader の評価関数
import evaluate


def qa_em_f1_with_evaluate(predict_text, gold_texts):
    """
    predict_text と gold_texts から SQuAD 形式の EM/F1 を計算して返す関数。

    predict_text : str
        モデルの予測回答テキスト
    gold_texts : List[str]
        正解候補の文字列リスト（1件以上）

    Returns
    -------
    Dict[str, float]
        {
            "exact_match": ... ,  # 0.0〜100.0
            "f1": ...             # 0.0〜100.0
        }
    """
    if not gold_texts:
        raise ValueError("gold_texts は 1 件以上必要です。")

    metric = evaluate.load("squad")

    predictions = [
        {
            "id": "example-1",
            "prediction_text": predict_text,
        }
    ]

    references = [
        {
            "id": "example-1",
            "answers": {
                "text": gold_texts,
                "answer_start": [0] * len(gold_texts),
            },
        }
    ]

    return metric.compute(predictions=predictions, references=references)

In [ ]:
# Reader の評価例
pred = "about 6000 hours"
label = ["6000 hours"]
metric = qa_em_f1_with_evaluate(pred, label)
print(f"EM: {metric['exact_match']:.2f}")
print(f"F1: {metric['f1']}")

In [ ]:
# F1 が悪い評価例
pred = "about 6000 dollars"
metric = qa_em_f1_with_evaluate(pred, label)
print(f"EM: {metric['exact_match']:.2f}")
print(f"F1: {metric['f1']}")

In [ ]:
## Reader 評価
@component
class Top1ItemEvaluateWithSquad:
    """
    前提:
    - answers_by_item は { item_id: [gold1, gold2, ...] } 形式
    - predicted_answers は Reader が返す answers リスト
    - documents は Retriever が返す documents リスト（doc.id と doc.meta.item_id を使う）
    """

    def __init__(self, missing_as_zero: bool = True):
        self.missing_as_zero = missing_as_zero

    @component.output_types(
        top_1_em=float,          # 0.0-100.0
        top_1_f1=float,          # 0.0-100.0
        evaluated_items=int,
        total_items=int,
        per_item=dict,
    )
    def run(self, answers_by_item: dict, predicted_answers: list, documents: list):
        # 1) doc.id -> item_id の対応を作る
        docid_to_item = {}
        for d in documents:
            doc_id = str(getattr(d, "id", "") or "").strip()
            item_id = str((getattr(d, "meta", {}) or {}).get("item_id", "")).strip()
            if doc_id and item_id:
                docid_to_item[doc_id] = item_id

        # 2) item_id ごとに予測 top1 を選ぶ（score 最大）
        best_pred_by_item = {}
        best_score_by_item = {}

        for a in predicted_answers:
            pred_text = str(getattr(a, "data", "") or "").strip()
            if not pred_text:
                continue

            score = float(getattr(a, "score", 0.0) or 0.0)
            doc_id = str(getattr(a, "document_id", "") or "").strip()

            # 原則: document_id から item_id を引く
            item_id = docid_to_item.get(doc_id, "")

            # フォールバック: answer.meta に item_id があれば使う
            if not item_id:
                item_id = str((getattr(a, "meta", {}) or {}).get("item_id", "")).strip()

            if not item_id:
                continue

            prev = best_score_by_item.get(item_id, None)
            if prev is None or score > prev:
                best_score_by_item[item_id] = score
                best_pred_by_item[item_id] = pred_text

        # 3) item 単位で EM/F1 を計算（qa_em_f1_with_evaluate を利用）
        per_item = {}
        em_list = []
        f1_list = []

        for item_id, gold_texts in answers_by_item.items():
            pred_text = best_pred_by_item.get(item_id, "")

            if not pred_text:
                if self.missing_as_zero:
                    em = 0.0
                    f1 = 0.0
                else:
                    # 欠損 item は平均から除外
                    continue
            else:
                scores = qa_em_f1_with_evaluate(pred_text, gold_texts)
                em = float(scores["exact_match"])   # 0.0-100.0
                f1 = float(scores["f1"])            # 0.0-100.0

            per_item[item_id] = {
                "prediction": pred_text,
                "gold_answers": gold_texts,
                "exact_match": em,
                "f1": f1,
            }
            em_list.append(em)
            f1_list.append(f1)

        evaluated_items = len(em_list)
        total_items = len(answers_by_item)

        top_1_em = sum(em_list) / evaluated_items if evaluated_items else 0.0
        top_1_f1 = sum(f1_list) / evaluated_items if evaluated_items else 0.0

        return {
            "top_1_em": top_1_em,
            "top_1_f1": top_1_f1,
            "evaluated_items": evaluated_items,
            "total_items": total_items,
            "per_item": per_item,
        }

In [ ]:
# Reader 評価パイプライン
## 実行に非常に時間がかかる
## Retriever
qa_retriever = ElasticsearchBM25Retriever(document_store=document_store)

## Reader
from haystack.components.readers import ExtractiveReader
max_seq_length, doc_stride = 384, 128
model_ckpt = "deepset/minilm-uncased-squad2"
reader = ExtractiveReader(model=model_ckpt,
                          max_seq_length=max_seq_length,
                          stride=doc_stride)

# 1) パイプライン構築
reader_pipe = Pipeline()
reader_pipe.add_component("QAretriever", qa_retriever)
reader_pipe.add_component("QAlabel_agg", LabelAggregatorByItem(label_store))
reader_pipe.add_component("QAreader", reader)
reader_pipe.add_component("eval_top1", Top1ItemEvaluateWithSquad(missing_as_zero=True))

# 2) 接続
reader_pipe.connect("QAretriever.documents", "QAreader.documents")
reader_pipe.connect("QAlabel_agg.answers_by_item", "eval_top1.answers_by_item")
reader_pipe.connect("QAreader.answers", "eval_top1.predicted_answers")
reader_pipe.connect("QAretriever.documents", "eval_top1.documents")

# 3) 実行
n_questions = 0
sum_q_em = 0.0
sum_q_f1 = 0.0

n_items_total = 0
sum_item_em = 0.0
sum_item_f1 = 0.0

for i, row in dfs["test"].iterrows():
    print(f"iteration: {i}")
    question_id = str(row["id"])
    query = safe_str(row["question"])

    out = reader_pipe.run(
        {
            "QAlabel_agg": {"question_id": question_id},
            "QAretriever": {"query": query},
            "QAreader": {"query": query, "top_k": 20},
        }
    )

    ev = out["eval_top1"]

    n_questions += 1
    sum_q_em += float(ev["top_1_em"])
    sum_q_f1 += float(ev["top_1_f1"])

    for item_id, d in ev["per_item"].items():
        sum_item_em += float(d["exact_match"])
        sum_item_f1 += float(d["f1"])
        n_items_total += 1

# 4) スコア集計
macro_top_1_em = sum_q_em / n_questions if n_questions else 0.0
macro_top_1_f1 = sum_q_f1 / n_questions if n_questions else 0.0
micro_top_1_em = sum_item_em / n_items_total if n_items_total else 0.0
micro_top_1_f1 = sum_item_f1 / n_items_total if n_items_total else 0.0

print("macro_top_1_em:", macro_top_1_em)
print("macro_top_1_f1:", macro_top_1_f1)
print("micro_top_1_em:", micro_top_1_em)
print("micro_top_1_f1:", micro_top_1_f1)
print("questions:", n_questions)
print("items:", n_items_total)

In [ ]:
# スコアプロット
metrics_data = {
    'Metric': ['Macro Top-1 EM', 'Macro Top-1 F1'],
    'Value': [macro_top_1_em, macro_top_1_f1]
}
metrics_df = pd.DataFrame(metrics_data)

def plot_reader_metrics(metric_df):
  fig, ax = plt.subplots()
  metric.plot(kind='bar', x='Metric', y='Value', ax=ax)
  ax.set_xticklabels(["EM", "F1"])
  plt.legend(loc='upper left')
  plt.show()

plot_reader_metrics(metrics_df)